# Install

# By Khushi Shah

In [1]:
!pip install -q groq

# Imports and API Key

In [4]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

# works from local cloned repo
cwd = Path.cwd()
env_path = cwd / "objection-your-honor" / ".env"
if not env_path.exists():
    env_path = cwd / ".env"

load_dotenv(env_path, override=True)

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError(f"GROQ_API_KEY not found. Looked for .env at: {env_path}")

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

VALID_BIASES = {"sunk_cost", "anchoring", "confirmation_bias", "loss_aversion", "none"}

# System prompt

In [5]:
SYSTEM_PROMPT = """You are a cognitive bias classifier. You read a person's journal entry about a decision they are weighing and identify which cognitive bias, if any, is shaping their reasoning.

BIAS DEFINITIONS:
- sunk_cost: Justifying a choice by past, unrecoverable investment (time, money, effort) instead of future outcomes. Looks BACKWARD.
- anchoring: Judging value or reasonableness based on an initial reference number (original price, first offer, first quote) that may be arbitrary or irrelevant.
- confirmation_bias: Favoring evidence that supports an existing belief while ignoring, dismissing, or never seeking contrary evidence.
- loss_aversion: Weighing a potential loss much more heavily than an equal or larger potential gain, often preferring the safe or current option out of fear of losing something. Looks FORWARD.
- none: The reasoning is balanced, weighs trade-offs fairly, or no clear bias is present.

RULES:
1. Choose exactly one primary bias. Do NOT force a bias. If the reasoning is sound, return "none".
2. If a second bias is clearly present, put it in secondary_bias. Otherwise null.
3. trigger_phrases must be EXACT verbatim quotes copied from the entry. Max 3. Empty list if bias is "none".
4. confidence_score is between 0 and 1. Use 0.85+ only when the bias is explicit. Use 0.5-0.7 for ambiguous entries. Use below 0.5 if guessing.
5. reasoning is 1-2 sentences, referencing the trigger phrases.
6. Respond with ONLY a valid JSON object with keys: bias_type, secondary_bias, confidence_score, trigger_phrases, reasoning."""

# Few-shot examples

In [6]:
FEW_SHOT = [
    ("I've already spent six months and about 40k on this coding bootcamp. I hate it and the placement stats look bad, but I can't quit now, that would mean all of it was wasted.",
     {"bias_type": "sunk_cost", "secondary_bias": None, "confidence_score": 0.93,
      "trigger_phrases": ["already spent six months and about 40k", "can't quit now", "all of it was wasted"],
      "reasoning": "The decision to continue rests entirely on past investment rather than the bootcamp's future value, even though the person admits it is not working."}),

    ("We've been together four years. Things haven't been good for a while, but leaving now would make those four years pointless.",
     {"bias_type": "sunk_cost", "secondary_bias": None, "confidence_score": 0.82,
      "trigger_phrases": ["leaving now would make those four years pointless"],
      "reasoning": "Staying is justified by time already invested, not by whether the relationship will be good going forward."}),

    ("The laptop was originally 1,20,000 and now it's 85,000, so it's a massive deal. I should grab it before the price goes back up.",
     {"bias_type": "anchoring", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": ["originally 1,20,000", "it's a massive deal"],
      "reasoning": "The original price is acting as the reference point for value, so the discount feels big regardless of whether 85,000 is fair for the specs."}),

    ("My first offer was 6 LPA, so when they came back with 7 it felt like a big jump and I think I should just accept.",
     {"bias_type": "anchoring", "secondary_bias": None, "confidence_score": 0.85,
      "trigger_phrases": ["first offer was 6 LPA", "felt like a big jump"],
      "reasoning": "The first offer is setting the baseline, making 7 LPA feel generous without comparing it to market rates."}),

    ("I always knew my manager hates me. Today she skipped my update in the meeting, which proves it. The three times she praised me were just her being polite.",
     {"bias_type": "confirmation_bias", "secondary_bias": None, "confidence_score": 0.94,
      "trigger_phrases": ["I always knew my manager hates me", "which proves it", "just her being polite"],
      "reasoning": "A single ambiguous event is read as proof while contradicting evidence (the praise) is explained away."}),

    ("I'm sure this stock will 10x. I only follow people who are bullish on it, and the bearish analysts clearly don't understand the tech.",
     {"bias_type": "confirmation_bias", "secondary_bias": None, "confidence_score": 0.91,
      "trigger_phrases": ["I only follow people who are bullish on it", "bearish analysts clearly don't understand the tech"],
      "reasoning": "The person filters their information sources to match their belief and dismisses opposing views without engaging them."}),

    ("The new job pays 40% more and the team looks better, but I'd lose my seniority and my stable position here. I can't risk losing what I have, even if the upside is bigger.",
     {"bias_type": "loss_aversion", "secondary_bias": None, "confidence_score": 0.92,
      "trigger_phrases": ["can't risk losing what I have", "even if the upside is bigger"],
      "reasoning": "The person explicitly admits the upside is larger but lets fear of losing current status dominate the decision."}),

    ("I know index funds would grow more, but seeing my savings dip even 5% in a bad month would hurt way more than the gains would feel good, so I'm keeping everything in the FD.",
     {"bias_type": "loss_aversion", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": ["dip even 5%", "hurt way more than the gains would feel good"],
      "reasoning": "The person describes losses as feeling disproportionately worse than equivalent gains and chooses the safe option because of it."}),

    ("I'm choosing between two flats. One is 2k cheaper but a 40 minute commute, the other is closer. I made a table of rent, commute time, and lease flexibility, and since time matters most to me right now, the closer one wins.",
     {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.9,
      "trigger_phrases": [],
      "reasoning": "The person compares trade-offs against stated priorities and reaches a conclusion without leaning on any single distorting reference."}),

    ("I've already put two years into this startup and I can't risk walking away with nothing. Anyone who says it's failing just doesn't get our vision.",
     {"bias_type": "sunk_cost", "secondary_bias": "confirmation_bias", "confidence_score": 0.8,
      "trigger_phrases": ["already put two years into this startup", "just doesn't get our vision"],
      "reasoning": "The main justification is past investment, and criticism is dismissed rather than evaluated, which points to confirmation bias as a secondary factor."}),
]

# Helper functions

In [7]:
def build_messages(entry: str) -> list:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for example_entry, example_output in FEW_SHOT:
        messages.append({"role": "user", "content": example_entry})
        messages.append({"role": "assistant", "content": json.dumps(example_output)})
    messages.append({"role": "user", "content": entry})
    return messages


def verify_triggers(entry: str, phrases: list) -> list:
    """Keep only phrases that actually appear in the entry (case-insensitive)."""
    entry_lower = entry.lower()
    return [p for p in phrases if p.lower() in entry_lower]

# Classifier

In [8]:
def classify_entry(entry: str, retries: int = 1) -> dict:
    entry = entry.strip()
    if len(entry) < 20:
        return {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.0,
                "trigger_phrases": [], "reasoning": "Entry too short to analyze."}

    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=build_messages(entry),
                temperature=0,
                response_format={"type": "json_object"},
            )
            result = json.loads(response.choices[0].message.content)

            if result.get("bias_type") not in VALID_BIASES:
                raise ValueError("Invalid bias_type")
            if result.get("secondary_bias") not in (VALID_BIASES - {"none"}) | {None}:
                result["secondary_bias"] = None

            claimed = result.get("trigger_phrases", [])
            verified = verify_triggers(entry, claimed)

            # Penalize confidence when quoted phrases are not found in the entry
            model_conf = float(result.get("confidence_score", 0.5))
            if result["bias_type"] != "none" and claimed:
                ratio = len(verified) / len(claimed)
                model_conf = model_conf * (0.5 + 0.5 * ratio)
            elif result["bias_type"] != "none" and not claimed:
                model_conf *= 0.5

            result["trigger_phrases"] = verified
            result["confidence_score"] = round(min(max(model_conf, 0.0), 1.0), 2)
            return result

        except (json.JSONDecodeError, ValueError, KeyError):
            if attempt == retries:
                return {"bias_type": "none", "secondary_bias": None, "confidence_score": 0.0,
                        "trigger_phrases": [], "reasoning": "Classification failed after retry."}

# Test harness

In [9]:
tests = [
    ("I've already invested three years in this degree and quitting now would feel like a waste, even though I don't like the field anymore.", "sunk_cost"),
    ("The jacket was 5000, now it's 2999, so I'm basically stealing it and I have to buy it.", "anchoring"),
    ("Everyone who criticizes my plan is just jealous. The one article that supports me proves I'm right.", "confirmation_bias"),
    ("I got a better offer, but what if the startup shuts down in a year? I'd rather keep my safe job.", "loss_aversion"),
    ("I listed pros and cons for both cities, checked cost of living, and picked the one closer to family.", "none"),
]

correct = 0
for entry, expected in tests:
    out = classify_entry(entry)
    match = out["bias_type"] == expected
    correct += match
    print(f"{'PASS' if match else 'FAIL'} | expected={expected} got={out['bias_type']} conf={out['confidence_score']}")
    print(f"      triggers={out['trigger_phrases']}")

print(f"\n{correct}/{len(tests)} correct")

PASS | expected=sunk_cost got=sunk_cost conf=0.92
      triggers=['already invested three years', 'quitting now would feel like a waste']
PASS | expected=anchoring got=anchoring conf=0.92
      triggers=['The jacket was 5000', "now it's 2999", "so I'm basically stealing it"]
PASS | expected=confirmation_bias got=confirmation_bias conf=0.93
      triggers=['Everyone who criticizes my plan is just jealous', "The one article that supports me proves I'm right"]
PASS | expected=loss_aversion got=loss_aversion conf=0.92
      triggers=['what if the startup shuts down in a year', "I'd rather keep my safe job"]
PASS | expected=none got=none conf=0.9
      triggers=[]

5/5 correct


# Try your own entry

In [10]:
entry = "I've already spent two years preparing for this exam, so I can't stop now even though I'm burnt out."
print(json.dumps(classify_entry(entry), indent=2))

{
  "bias_type": "sunk_cost",
  "secondary_bias": null,
  "confidence_score": 0.94,
  "trigger_phrases": [
    "already spent two years preparing for this exam",
    "can't stop now"
  ],
  "reasoning": "The decision to continue is justified solely by the time already invested, ignoring current burnout and future outcomes."
}


----